# Aula 12 — Classificação

Na Aula 11 o modelo estimava um **número**: a taxa de engajamento de um post. Hoje o alvo muda de natureza. Em vez de "quanto", a pergunta vira "**qual categoria**": esse post vai viralizar ou não?

Isso se chama **classificação**, e quase tudo do esqueleto continua igual: features `X`, alvo `y`, `train_test_split`, `fit`, `predict`. O que muda, e onde mora a atenção da aula:

1. **O rótulo é uma decisão sua.** "Viralizou" não existe nos dados. Você define, com um corte, o que conta como viral, e essa escolha muda todo o resultado.
2. **Acurácia engana.** Quando "viralizou" é raro (digamos, 1 post em 10), um modelo que nunca aposta em viral acerta 90% e não serve para nada. Precisamos de medidas melhores: precisão, recall, F1, e a matriz de confusão.
3. **O corte de decisão pode ser afinado.** O modelo devolve uma probabilidade; você escolhe a partir de que probabilidade chama de "viral", e isso é um trade-off explícito.

## 1. Preparando o ambiente

Mesmas bibliotecas da Aula 11: `pandas`, `scikit-learn` e `matplotlib`. Dentro da pasta `aulas/12-classificacao/`:

No Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, use `pip install -r requirements.txt`.

## 2. Carregar a coleta e as features

Reaproveitamos exatamente o que foi feito na Aula 11: carregar a exportação, remover duplicata, construir features derivadas **sem vazamento** (nada de `likes`, `comments`, `shares`, `plays` como entrada). Se algum trecho aqui parecer novo, volte na Aula 11.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

df = pd.read_csv("dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy() # lembram do pq? :)

# features "de antes da publicação", iguais às da Aula 11
X = pd.DataFrame(index=df.index)
X["seguidores_autor"] = df["author_followers"]
X["videos_autor"] = df["author_videos"]
X["tam_legenda"] = df["body"].fillna("").str.len()
padrao_emoji = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")
X["n_emojis"] = df["body"].fillna("").apply(lambda s: len(padrao_emoji.findall(s)))
X["n_hashtags"] = df["hashtags"].fillna("").apply(lambda s: 0 if s == "" else len(s.split(",")))
momento = pd.to_datetime(df["timestamp"])
X["hora"] = momento.dt.hour
X["dia_semana"] = momento.dt.dayofweek

print(f"Posts: {len(df)}  |  Features: {list(X.columns)}")

## 3. O rótulo é uma decisão: definindo "viralizou"

Não existe coluna `viralizou`. A gente **cria** uma, a partir de um corte no número de visualizações. A escolha do corte é metodológica e precisa ficar registrada, porque muda tudo: um corte alto deixa pouquíssimos posts como "viral" (classe rara, difícil de prever); um corte baixo deixa quase todo mundo como "viral" (aí o rótulo perde sentido).

Vamos usar o **percentil 90** de `plays` como corte: os 10% de posts mais vistos da própria coleta contam como "viralizou".

In [ ]:
corte = df["plays"].quantile(0.90)  # o valor de plays que separa os 10% mais vistos
y = (df["plays"] > corte).astype(int)  # 1 = viralizou, 0 = não

print(f"Corte (percentil 90 de plays): {int(corte):,} visualizações")
print(f"Posts marcados como 'viralizou': {y.sum()} de {len(y)}  ({y.mean():.0%})")

# como o corte muda a proporção de positivos
for p in [0.75, 0.80, 0.90, 0.95]:
    c = df["plays"].quantile(p)
    print(f"  percentil {int(p*100)}: corte {int(c):>10,}  ->  {(df['plays'] > c).mean():.0%} viram 'viral'")

**O que observar:** com o corte no percentil 90, cerca de 10% dos posts são "viral". Essa é uma classe **desbalanceada**: muito mais 0 do que 1. Guarde esse número, ele é a chave para ler a acurácia na Seção 6. E repare que mexer no percentil muda a proporção de forma direta; o corte não é um detalhe técnico, é parte da resposta.

## 4. Treino, teste e a estratificação

Igual à Aula 11, separamos treino e teste. A novidade é `stratify=y`: com classe rara, um sorteio comum pode, por azar, colocar quase todos os "viral" no treino e quase nenhum no teste (ou o contrário). `stratify=y` força a mesma proporção de 0 e 1 dos dois lados.

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y,  # mantém a proporção viral/não-viral igual no treino e no teste
)

print(f"Treino: {len(y_treino)} posts, {y_treino.mean():.0%} viral")
print(f"Teste:  {len(y_teste)} posts, {y_teste.mean():.0%} viral")

## 5. Regressão logística: `predict` e `predict_proba`

`LogisticRegression`, apesar do nome, é um modelo de **classificação**. Ela estima, para cada post, a **probabilidade** de ser da classe 1 (viral). Depois transforma essa probabilidade em uma decisão: probabilidade acima de 0,5, chama de viral; abaixo, não.

- `.predict(X)` devolve a decisão já pronta (0 ou 1).
- `.predict_proba(X)` devolve as probabilidades. A coluna 1 é a probabilidade de ser viral.

`max_iter=1000` só dá mais rodadas para o algoritmo convergir; sem isso ele às vezes reclama.

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_treino, y_treino)

decisao = modelo.predict(X_teste)  # 0 ou 1, já com corte em 0,5
probabilidade = modelo.predict_proba(X_teste)[:, 1]  # probabilidade de ser viral

# espiando os 5 primeiros posts de teste
pd.DataFrame({
    "prob_viral": probabilidade[:5].round(3),
    "decisao": decisao[:5],
    "real": y_teste.values[:5],
})

## 6. Matriz de confusão e por que a acurácia engana

A **matriz de confusão** conta os quatro tipos de resultado:

|  | modelo disse "não" | modelo disse "viral" |
|---|---|---|
| **era "não"** | verdadeiro negativo (VN) | falso positivo (FP) |
| **era "viral"** | falso negativo (FN) | verdadeiro positivo (VP) |

A **acurácia** é `(VN + VP) / total`: a fração de acertos. Parece uma boa medida, mas com classe rara ela mente. Veja:

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

cm = confusion_matrix(y_teste, decisao)
print("Matriz de confusão:")
print(f"                 modelo: não   modelo: viral")
print(f"  era não:        {cm[0,0]:>6}        {cm[0,1]:>6}")
print(f"  era viral:      {cm[1,0]:>6}        {cm[1,1]:>6}")
print()
print(f"Acurácia:  {accuracy_score(y_teste, decisao):.2f}")
print(f"Precisão:  {precision_score(y_teste, decisao, zero_division=0):.2f}   (dos que o modelo chamou de viral, quantos eram)")
print(f"Recall:    {recall_score(y_teste, decisao, zero_division=0):.2f}   (dos virais de verdade, quantos o modelo pegou)")
print(f"F1:        {f1_score(y_teste, decisao, zero_division=0):.2f}   (média equilibrada de precisão e recall)")

**O que observar:** a acurácia fica alta (perto de 0,90), mas olhe a linha "era viral": o modelo quase não marca nenhum post como viral, então o **recall** é baixíssimo. Ele acerta 90% porque 90% dos posts não são virais e ele chuta "não" para quase todo mundo. Um modelo que responde sempre "não viraliza" teria acurácia ~90% e seria inútil. É por isso que, com classe rara, a acurácia sozinha não diz nada; precisão e recall é que contam a história.

## 7. Ajustando o corte de decisão (*threshold*)

O modelo decidiu com o corte padrão de 0,5 na probabilidade. Mas 0,5 não é sagrado. Se pegar mais virais importa mais do que evitar alarme falso (por exemplo, para uma equipe de conteúdo que quer *não deixar passar* um post promissor), a gente **abaixa o corte**: qualquer post com probabilidade acima de, digamos, 0,20 já é chamado de viral.

Isso é um trade-off explícito: corte mais baixo → mais recall (pega mais virais), menos precisão (mais alarme falso).

In [ ]:
for t in [0.50, 0.30, 0.20, 0.15]:
    decisao_t = (probabilidade >= t).astype(int)
    p = precision_score(y_teste, decisao_t, zero_division=0)
    r = recall_score(y_teste, decisao_t, zero_division=0)
    f = f1_score(y_teste, decisao_t, zero_division=0)
    vp = int(((decisao_t == 1) & (y_teste.values == 1)).sum())
    fp = int(((decisao_t == 1) & (y_teste.values == 0)).sum())
    print(f"corte {t:.2f}:  precisão {p:.2f}  recall {r:.2f}  F1 {f:.2f}   (pegou {vp} virais, {fp} alarmes falsos)")

**O que observar:** conforme o corte cai, o modelo pega mais posts virais (recall sobe), mas erra mais para o outro lado (precisão cai). Não existe corte "certo": existe o corte que faz sentido para o que você vai fazer com a previsão. Escolher esse corte é decisão sua, não do `scikit-learn`.

## 8. Uma árvore de classificação e a importância das features

Como na Aula 11, uma **árvore** capta relações não-lineares que a logística não pega. E ela expõe `feature_importances_`: o quanto cada feature contribuiu para as decisões (isso não é o mesmo que coeficiente, mas serve para ver o que o modelo mais usou).

`class_weight="balanced"` diz para a árvore dar mais peso aos poucos exemplos virais, para ela não ignorar a classe rara.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arvore = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
arvore.fit(X_treino, y_treino)
decisao_arvore = arvore.predict(X_teste)

print("Árvore (profundidade 4, class_weight balanced):")
print(f"  precisão {precision_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  recall {recall_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  F1 {f1_score(y_teste, decisao_arvore, zero_division=0):.2f}")
print()

importancias = pd.DataFrame({
    "feature": X.columns,
    "importancia": arvore.feature_importances_,
}).sort_values("importancia", ascending=False)
importancias

**O que observar:** com `class_weight="balanced"` a árvore passa a apostar em viral com mais frequência, o recall sobe bastante e a precisão cai (o mesmo trade-off da Seção 7, por outro caminho). As `feature_importances_` mostram quais features a árvore mais usou para separar viral de não-viral, nesta coleta. Igual ao coeficiente da Aula 11: isso descreve o que o modelo usou nos dados que viu, não uma regra causal de como viralizar.

## 9. Quando der errado

- **Acurácia alta, mas o modelo nunca acerta a classe rara.** É o caso da Seção 6. Sempre olhe a matriz de confusão e o recall da classe que te interessa, não só a acurácia.
- **`precision`/`recall` deram `0` e um aviso de "ill-defined".** O modelo não previu nenhum positivo (ou não havia positivos no teste). Reduza o threshold, use `class_weight="balanced"`, ou confira se o `stratify=y` está no `train_test_split`.
- **Só uma classe no treino ou no teste.** O corte ficou extremo demais (quase ninguém é viral, ou quase todo mundo é). Reveja o percentil da Seção 3.
- **Usei `predict` quando queria a probabilidade (ou o contrário).** `.predict` já devolve 0/1 com corte fixo em 0,5; `.predict_proba(X)[:, 1]` devolve a probabilidade, que é o que você precisa para mexer no threshold.
- **`ValueError: could not convert string to float` / `Input contains NaN`.** Igual à Aula 11: coluna de texto no `X` precisa virar número (`pd.get_dummies`), valor ausente precisa de decisão explícita antes do `fit`.
- **`ModuleNotFoundError: No module named 'sklearn'`.** Ambiente não criado ou dependências não instaladas. `uv venv .venv` e `uv pip install -r requirements.txt` dentro de `aulas/12-classificacao/`.

## 10. Prática: faça agora

Abra `exercicios/exercicio-12-classificacao.ipynb`. Na sua coleta, você vai **definir e justificar o próprio corte** de "viralizou" (não precisa ser o percentil 90), treinar a logística, ler a matriz de confusão, testar pelo menos dois thresholds diferentes e comparar com uma árvore. No README, você registra o corte escolhido, por que ele faz sentido, e uma frase sobre a favor de quem o seu modelo erra (ele deixa passar muitos virais? ou dá muito alarme falso?).